# Run the real pipeline

Unlike `build_demo_db.ipynb` (hand-authored, illustrative data, no network
calls), this notebook calls `protein_selector.pipeline.run_pipeline` for
real: real RCSB hard-filters search, real simulability/composition checks,
real ligand CCD/SMILES lookup, real RDKit (and Meeko, if installed)
parameterizability, real Europe PMC literature counts, and a real AlphaFold
DB lookup for ex02 -- every number below comes from an actual API response,
nothing is invented.

**Stage config is grouped into dataclasses** (`HardFilterConfig`, `MdSimulationConfig`,
`PocketDetectionConfig`, ...) -- see `pipeline.py`'s own docstring for the full
list and what each corresponds to (`ex02`/`ex03`/`ex04` are still the internal
persisted-column labels, but the pipeline's own API and config classes describe
what each stage actually does instead of that jargon).

MD simulation and pocket detection need the conda-only
`environment-validation.yml` env (a local `fpocket` binary too, for pocket
detection) -- this notebook enables both below, so it must be run through that
conda env's Python (the "Python 3 (protein-selector-validation, conda)" kernel),
not the base venv. Docking (`ex04`, real Vina + PLIP) still isn't wired into the
pipeline at all -- no receptor-prep/pocket-center-extraction code exists yet to
auto-run it, so it will always show `"not_run"` regardless of config.

`max_candidates` is kept small (a real Meeko 3D-embedding step, if
installed, can be slow for some real bound ligands) -- raise it once you've
confirmed a run completes in reasonable time for your machine.

Requires the `notebook` dependency group: `uv sync --group notebook`.

In [1]:
import logging
from pathlib import Path

import pandas as pd

from protein_selector.pipeline import (
    HardFilterConfig,
    MdSimulationConfig,
    PocketDetectionConfig,
    run_pipeline,
)

logging.basicConfig(level=logging.INFO, format="%(message)s")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Separate db from build_demo_db.ipynb's illustrative one, so the two never mix.
DB_PATH = Path("cache/protein_selector_real.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
CSV_PATH = Path("report_real.csv")

/home/yescalona/.local/share/micromamba/envs/protein-selector-validation/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
rows = run_pipeline(
    db_path=DB_PATH,
    hard_filters=HardFilterConfig(max_candidates=100, random_seed=1),
    # modeling_lookup left at its default (ModelingLookupConfig(enabled=True))
    # -- the real AlphaFold DB fetch-only check, persisted as exercise "ex02".
    md_simulation=MdSimulationConfig(
        enabled=True,
        n_steps=50,
        max_minimization_iterations=10,
        max_residues=50,  # candidates bigger than this skip MD entirely --
        # see MdSimulationConfig's own docstring for why residues, not atoms.
    ),
    pocket_detection=PocketDetectionConfig(enabled=True),
    report_csv_path=CSV_PATH,
)
len(rows)

TypeError: run_pipeline() got an unexpected keyword argument 'max_minimization_iterations'

## The real joined report

In [ ]:
from protein_selector.core.report import rows_to_dataframe

report_df = rows_to_dataframe(rows)
report_df

## What's real here, spelled out

- `title`/`organism`/`n_residues`/`resolution`/... -- real RCSB entry metadata.
- `ligand_parameterizable` -- a real RDKit sanitization result (and real Meeko
  3D-embed + PDBQT-write result, if the `validate` extra is installed).
- `litref_count` -- a real Europe PMC hit count for this exact PDB ID.
- `ex02_status`/`ex02_predicted_difficulty` -- a real AlphaFold DB lookup: if
  the candidate's UniProt accession has a modeled entry, this reflects its
  actual published confidence fractions; if not, `ex02_status` is `"fail"`
  with `FailureMode.COMPLETENESS`, not guessed.

In [ ]:
report_df[["pdb_id", "uniprot_id", "ligand_ccd", "ligand_parameterizable", "litref_count", "ex02_status", "ex02_predicted_difficulty", "ex02_failure_mode"]]